In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Notebook 11: Computational Efficiency

- Training time: from master_results_all.csv
- Inference timing: 5 warm-up runs (discarded) + 100 timed runs on the full test set
- Mean and standard deviation computed over the 100 timed runs only
- Per-flow latency and throughput derived from the mean
- Model size: file size on disk
- Hardware: from /proc/cpuinfo
- Canonical output: `results/efficiency_table.csv`
- Raw per-run timings: `results/raw_efficiency_timings.json`

In [2]:
import time
import numpy as np
import pandas as pd
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [3]:
# CPU info
cpu_info = {}
with open('/proc/cpuinfo') as f:
    for line in f:
        if line.startswith('model name') and 'model_name' not in cpu_info:
            cpu_info['model_name'] = line.split(':')[1].strip()
        if line.startswith('cpu MHz') and 'cpu_mhz' not in cpu_info:
            cpu_info['cpu_mhz'] = line.split(':')[1].strip()
cpu_info['physical_cores'] = os.cpu_count()
print('CPU:', cpu_info)

CPU: {'model_name': 'AMD EPYC 9354P 32-Core Processor', 'cpu_mhz': '3249.984', 'physical_cores': 2}


In [4]:
master = pd.read_csv('results/master_results_all.csv')

# Load test sets
ugr_test = pd.read_csv('data/processed/ugr_test.csv')
X_te_ugr = ugr_test.drop(columns=['Prediction'])

cic_test = pd.read_csv('data/processed/cic_test.csv')
X_te_cic = cic_test.drop(columns=['label', 'label_binary'])

MODELS = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression']
WARMUP_RUNS = 5    # discarded, to stabilise CPU caches
TIMING_RUNS = 100  # measured

rows = []
raw_timings = []
for dataset, prefix, X_te in [
    ('UGRansome2024', 'ugr', X_te_ugr),
    ('CICIoT2023', 'cic', X_te_cic)
]:
    for model in MODELS:
        ext = 'joblib'
        model_path = f'results/models/{prefix}_{model}.{ext}'
        if not os.path.exists(model_path):
            alt = f'results/models/{prefix}_{model}.json'
            if os.path.exists(alt):
                model_path = alt
                ext = 'json'

        if not os.path.exists(model_path):
            print(f'  MISSING: {model_path}')
            continue

        clf = joblib.load(model_path)
        model_size_mb = round(os.path.getsize(model_path) / 1024 / 1024, 3)

        row_master = master[(master['dataset'] == dataset) & (master['model'] == model)]
        train_time = float(row_master['training_time_sec'].values[0]) if not row_master.empty else None

        n_samples = len(X_te)

        # Warm-up runs (discarded)
        for _ in range(WARMUP_RUNS):
            clf.predict(X_te)

        # Timed runs
        times = []
        for _ in range(TIMING_RUNS):
            t0 = time.perf_counter()
            clf.predict(X_te)
            times.append(time.perf_counter() - t0)

        mean_total_sec = float(np.mean(times))
        std_total_sec  = float(np.std(times))
        per_flow_us    = round(mean_total_sec / n_samples * 1e6, 4)  # microseconds
        throughput     = round(n_samples / mean_total_sec, 1)        # flows/sec

        rows.append({
            'dataset': dataset,
            'model': model,
            'training_time_sec': train_time,
            'inference_mean_sec': round(mean_total_sec, 6),
            'inference_std_sec': round(std_total_sec, 6),
            'per_flow_us': per_flow_us,
            'throughput_flows_per_sec': throughput,
            'model_size_mb': model_size_mb,
            'n_test_samples': n_samples,
            'warmup_runs': WARMUP_RUNS,
            'timing_runs': TIMING_RUNS,
        })
        raw_timings.append({
            'dataset': dataset,
            'model': model,
            'warmup_runs': WARMUP_RUNS,
            'timing_runs': TIMING_RUNS,
            'individual_times_sec': times,
        })
        print(f'{dataset} | {model}: {per_flow_us} us/flow, {throughput} flows/s, {model_size_mb} MB')

COLS = ['dataset','model','training_time_sec','inference_mean_sec','inference_std_sec',
        'per_flow_us','throughput_flows_per_sec','model_size_mb','n_test_samples',
        'warmup_runs','timing_runs']
df_eff = pd.DataFrame(rows)[COLS]
df_eff.to_csv('results/efficiency_table.csv', index=False)
print('\nSaved results/efficiency_table.csv (canonical)')

import json
with open('results/raw_efficiency_timings.json', 'w') as f:
    json.dump(raw_timings, f, indent=2)
print('Saved results/raw_efficiency_timings.json')

with open('results/cpu_info.json', 'w') as f:
    json.dump(cpu_info, f, indent=2)
print('Saved results/cpu_info.json')

df_eff

UGRansome2024 | RandomForest: 4.5673 us/flow, 218950.1 flows/s, 3.295 MB
UGRansome2024 | DecisionTree: 0.1031 us/flow, 9701598.3 flows/s, 0.015 MB


UGRansome2024 | XGBoost: 2.3252 us/flow, 430064.3 flows/s, 0.08 MB


UGRansome2024 | LogisticRegression: 1.2496 us/flow, 800275.2 flows/s, 0.001 MB


CICIoT2023 | RandomForest: 1.7357 us/flow, 576152.4 flows/s, 2.767 MB


CICIoT2023 | DecisionTree: 0.07 us/flow, 14280702.0 flows/s, 0.024 MB


CICIoT2023 | XGBoost: 1.2777 us/flow, 782654.7 flows/s, 0.051 MB


CICIoT2023 | LogisticRegression: 0.0479 us/flow, 20867001.7 flows/s, 0.001 MB

Saved results/efficiency_table.csv (canonical)
Saved results/raw_efficiency_timings.json
Saved results/cpu_info.json


,dataset,model,training_time_sec,inference_mean_sec,inference_std_sec,per_flow_us,throughput_flows_per_sec,model_size_mb,n_test_samples,warmup_runs,timing_runs
0,UGRansome2024,RandomForest,2.358,0.082083,0.008417,4.5673,218950.1,3.295,17972,5,100
1,UGRansome2024,DecisionTree,0.304,0.001852,0.000169,0.1031,9701598.3,0.015,17972,5,100
2,UGRansome2024,XGBoost,0.659,0.041789,0.008763,2.3252,430064.3,0.080,17972,5,100
3,UGRansome2024,LogisticRegression,0.478,0.022457,0.002120,1.2496,800275.2,0.001,17972,5,100
4,CICIoT2023,RandomForest,5.052,0.069417,0.010572,1.7357,576152.4,2.767,39995,5,100
5,CICIoT2023,DecisionTree,0.833,0.002801,0.000392,0.0700,14280702.0,0.024,39995,5,100
6,CICIoT2023,XGBoost,1.263,0.051102,0.005798,1.2777,782654.7,0.051,39995,5,100
7,CICIoT2023,LogisticRegression,6.871,0.001917,0.000744,0.0479,20867001.7,0.001,39995,5,100


In [5]:
print('Notebook 11 complete.')

Notebook 11 complete.
